In [1]:
# Cell 1 — Setup & imports
import polars as pl
from Python import statcast, pitcher_features

years = [2025]

In [2]:
# Cell 2 — Inspect vocabulary/constants
print(pitcher_features.PITCH_TYPES)
print(pitcher_features.CANON_PITCH)
print(pitcher_features.FANGRAPHS_FIP_CONSTANT)
print(pitcher_features.BUILD_COLUMNS)

('ff', 'si', 'fc', 'sl', 'st', 'cu', 'ch', 'fs')
{'FF': 'ff', 'FA': 'ff', 'SI': 'si', 'FT': 'si', 'FC': 'fc', 'SL': 'sl', 'SV': 'sl', 'ST': 'st', 'CU': 'cu', 'KC': 'cu', 'CS': 'cu', 'CH': 'ch', 'FS': 'fs', 'SF': 'fs'}
{2021: 3.17, 2022: 3.112, 2023: 3.255, 2024: 3.166, 2025: 3.135, 2026: 3.099}
('game_pk', 'game_date', 'player_name', 'pitcher', 'stand', 'p_throws', 'home_team', 'away_team', 'inning', 'inning_topbot', 'at_bat_number', 'pitch_number', 'pitch_type', 'type', 'description', 'events', 'bb_type', 'zone', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'release_extension', 'release_pos_x', 'release_pos_z', 'vy0', 'vz0', 'ay', 'az', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'bat_score', 'post_bat_score')


In [3]:
# Cell 3 — Load raw data with the columns pitcher_features actually needs
# (statcast.DEFAULT_COLUMNS is missing vy0/vz0/ay/az/bb_type/zone/etc.)
raw = statcast.load_statcast_years(years, columns=list(pitcher_features.BUILD_COLUMNS))
raw.shape, raw.columns

((712528, 35),
 ['game_pk',
  'game_date',
  'player_name',
  'pitcher',
  'stand',
  'p_throws',
  'home_team',
  'away_team',
  'inning',
  'inning_topbot',
  'at_bat_number',
  'pitch_number',
  'pitch_type',
  'type',
  'description',
  'events',
  'bb_type',
  'zone',
  'release_speed',
  'release_spin_rate',
  'pfx_x',
  'pfx_z',
  'release_extension',
  'release_pos_x',
  'release_pos_z',
  'vy0',
  'vz0',
  'ay',
  'az',
  'estimated_ba_using_speedangle',
  'estimated_woba_using_speedangle',
  'woba_value',
  'woba_denom',
  'bat_score',
  'post_bat_score'])

In [4]:
# Cell 4 — Pitch-level enrichment (VAA, canon_pitch, outs_on_play, run_delta)
pl_df = pitcher_features._pitch_level(raw)
pl_df.select(["pitch_type", "canon_pitch", "vaa", "ivb", "hb", "outs_on_play", "run_delta"]).head(10)

# Sanity check VAA range: should generally fall between -4 and -12 degrees for MLB pitches
pl_df.select(
    pl.col("vaa").min().alias("vaa_min"),
    pl.col("vaa").max().alias("vaa_max"),
    pl.col("vaa").mean().alias("vaa_mean"),
)

pl_df.group_by("canon_pitch").agg(
    pl.len().alias("n_pitches"),
    pl.col("vaa").mean().alias("vaa_mean"),
    pl.col("vaa").min().alias("vaa_min"),
    pl.col("vaa").max().alias("vaa_max"),
    pl.col("vaa").std().alias("vaa_std"),
).sort("vaa_mean")

canon_pitch,n_pitches,vaa_mean,vaa_min,vaa_max,vaa_std
str,u32,f64,f64,f64,f64
null,4568,-11.598579,-69.16554,-1.559577,4.954856
"""cu""",60664,-9.720522,-28.952296,-1.847021,1.405117
"""st""",54295,-7.717134,-13.354518,-2.880138,1.211664
"""sl""",105690,-7.69271,-30.489608,-0.162358,1.278272
"""fs""",23267,-7.636867,-13.173837,-2.26257,1.181278
"""ch""",73209,-7.459763,-13.922936,-1.19987,1.085882
"""fc""",53437,-6.114489,-12.6009,-1.164881,1.187431
"""si""",110184,-5.864897,-10.924388,-0.764315,0.98691
"""ff""",227214,-4.710301,-16.928486,0.366979,1.020438


In [5]:
# Cell 5 — Starter identification
starters = pitcher_features._starter_keys(pl_df)
starters.shape

# Confirm exactly one starter per (game_pk, inning_topbot) -- catches double-header
# or data anomalies silently producing wrong starter attribution
check = (
    pl_df.filter(pl.col("inning") == 1)
    .group_by(["game_pk", "inning_topbot"])
    .agg(pl.col("pitcher").n_unique().alias("n_pitchers_seen"))
)
starters.shape[0], check.shape[0]  # starters rows should equal number of (game_pk, inning_topbot) pairs

(4860, 4860)

In [6]:
# Cell 6 — Full aggregation: one row per starting-pitcher-game
starts = pitcher_features.build_pitcher_starts(raw, min_batters_faced=9)
starts.shape

# Uniqueness check -- should be 0 duplicates on the primary key
starts.select(["game_pk", "pitcher"]).is_duplicated().sum()

0

In [7]:
# Cell 7 — Inspect core label columns (K, PA, Outs) -- these are LABELS, not features
starts.select(["player_name", "game_date", "PA", "K", "BB", "HBP", "HR", "Outs", "wOBA", "xWOBA" if "xWOBA" in starts.columns else "xwOBA"]).head(10)

# Distribution sanity check on the eventual model target
starts_check = starts.with_columns((pl.col("K") / pl.col("PA")).alias("k_rate"))
starts_check.select(
    pl.col("k_rate").min().alias("k_rate_min"),
    pl.col("k_rate").max().alias("k_rate_max"),
    pl.col("k_rate").mean().alias("k_rate_mean"),
)

k_rate_min,k_rate_max,k_rate_mean
f64,f64,f64
0.0,0.722222,0.218136


In [8]:
# Cell 8 — Spot-check one known start end-to-end
starts.filter(pl.col("player_name").str.contains("Cristopher")).select(
    ["game_date", "PA", "K", "Outs", "wOBA", "xwOBA", "extension", "rel_x", "rel_z"]
)
# Replace "Skubal" with any pitcher name you want to manually verify against a box score

game_date,PA,K,Outs,wOBA,xwOBA,extension,rel_x,rel_z
date,u32,u32,i32,f64,f64,f64,f64,f64
2025-03-31,22,7,16,0.277273,0.33339,7.146237,1.950753,6.074409
2025-04-06,25,9,17,0.374,0.235812,7.155682,1.932386,6.097159
2025-04-12,26,3,20,0.378846,0.312692,6.948421,1.812632,5.923263
2025-04-17,28,12,21,0.294643,0.199355,6.994845,1.928041,6.039897
2025-04-22,11,2,5,0.486364,0.427454,7.25,1.938103,5.879828
…,…,…,…,…,…,…,…,…
2025-09-05,27,5,21,0.251852,0.201204,6.92439,1.903293,5.85378
2025-09-10,22,6,18,0.211364,0.376593,6.849462,1.886774,5.978065
2025-09-16,27,6,21,0.314815,0.335649,6.804902,1.817941,5.797647


In [9]:
# Cell 9 — Arsenal / pitch-type usage check
usage_cols = [f"{pt}_usage_vR" for pt in pitcher_features.PITCH_TYPES] + \
             [f"{pt}_usage_vL" for pt in pitcher_features.PITCH_TYPES]
starts.select(["player_name"] + usage_cols).head(5)

# Sanity check: usage rates vs each hand should sum to <= 1.0 (not necessarily exactly 1.0
# since unmapped pitch types like knuckleballs/eephus aren't in PITCH_TYPES)
vR_cols = [f"{pt}_usage_vR" for pt in pitcher_features.PITCH_TYPES]
starts.with_columns(pl.sum_horizontal(vR_cols).alias("total_usage_vR")).select(
    pl.col("total_usage_vR").min().alias("usage_vR_min"),
    pl.col("total_usage_vR").max().alias("usage_vR_max"),
)

usage_vR_min,usage_vR_max
f64,f64
0.0,1.0


In [10]:
# Cell 10 — Confirm no leaked helper columns survived
helper_prefixes = ("_pit_", "_ff_", "_si_", "_fc_", "_sl_", "_st_", "_cu_", "_ch_", "_topbot")
leaked = [c for c in starts.columns if c.startswith(helper_prefixes)]
leaked  # should be an empty list

[]

In [11]:
# Cell 11 — League HR/FB (must use ALL pitches, not just starters, per dev-notes)
hr_fb = pitcher_features.league_hr_fb_from_pitches(raw)
hr_fb

{2025: 0.13420746336017483}

In [12]:
# Cell 12 — FIP / xFIP enrichment
enriched = pitcher_features.add_fip_xfip(starts, league_hr_fb=hr_fb)
enriched.select(["player_name", "season", "FIP", "xFIP"]).sort("FIP").head(10)

# Sanity check: league-average FIP should land near the season's published cFIP constant
enriched.group_by("season").agg(pl.col("FIP").mean().alias("avg_FIP"), pl.col("xFIP").mean().alias("avg_xFIP"))

season,avg_FIP,avg_xFIP
i32,f64,f64
2025,4.368229,4.231049


In [13]:
# Cell 13 — Verify FIP core-only mode (include_constant=False) shifts every value by the constant
core_only = pitcher_features.add_fip_xfip(starts, league_hr_fb=hr_fb, include_constant=False)
diff_check = enriched.select("player_name", "game_pk", "season", "FIP").join(
    core_only.select("player_name", "game_pk", "season", "FIP"),
    on=["player_name", "game_pk", "season"],
    suffix="_core",
).with_columns((pl.col("FIP") - pl.col("FIP_core")).alias("diff"))

diff_check.select(
    pl.col("diff").min().alias("diff_min"),
    pl.col("diff").max().alias("diff_max"),
)

diff_min,diff_max
f64,f64
3.135,3.135
